# Multimodal Semantic Search — Training (Google Colab)

End-to-end training of the dual-encoder model on Colab's free T4. Uses the **pre-cached ResNet-50 features**, so each epoch is ~50 s and the full 20-epoch run finishes in ~15–20 min.

**Drive layout this notebook expects** (under `My Drive`):

```
MyDrive/dl-multimodal/
├── cached/                          ← 6 files: 3 *_image_features.pt + 3 *_id_to_idx.json
├── flickr30k-images/                ← optional, only needed if running the demo on Colab
├── splits/                          ← optional, identical to the repo's data/splits/
└── results.csv                      ← captions file (~12 MB)
```

Output (trained checkpoint, gallery index, loss curve) is written to `MyDrive/dl-multimodal/output/`.

**Runtime → Change runtime type → GPU (T4)** before running.

## 1. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Sanity-check that the data is where we expect

In [4]:
DRIVE = '/content/drive/MyDrive/dl-multimodal'
!ls -la {DRIVE}/cached
!ls -la {DRIVE}/raw

ls: cannot access '/content/drive/MyDrive/dl-multimodal/cached': No such file or directory
ls: cannot access '/content/drive/MyDrive/dl-multimodal/raw': No such file or directory


## 3. Clone the repo and install missing deps

In [5]:
%cd /content
!rm -rf dl-multimodal-semantic-search
!git clone https://github.com/nlklfor/dl-multimodal-semantic-search.git
%cd dl-multimodal-semantic-search
!git checkout dev   # latest integration branch (incl. PR #21 gallery script)

/content
Cloning into 'dl-multimodal-semantic-search'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 64 (delta 10), reused 60 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 179.44 KiB | 6.41 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/dl-multimodal-semantic-search
Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'


In [6]:
# Colab pre-installs torch / torchvision / pandas / matplotlib; only these are missing.
!pip install -q transformers tqdm

## 4. Wire Drive data into the project's expected paths

In [7]:
import os

DRIVE = '/content/drive/MyDrive/dl-multimodal'
os.makedirs('data/cached', exist_ok=True)
os.makedirs('data/raw',    exist_ok=True)

# Symlink each cached feature / json file (no copying — instant).
for fname in os.listdir(f'{DRIVE}/cached'):
    src, dst = f'{DRIVE}/cached/{fname}', f'data/cached/{fname}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

# Symlink captions (results.csv lives at the top level of dl-multimodal/, not under raw/).
if not os.path.exists('data/raw/results.csv'):
    os.symlink(f'{DRIVE}/results.csv', 'data/raw/results.csv')

# (Optional) wire raw images too, in case you want to launch the demo from Colab later.
if os.path.isdir(f'{DRIVE}/flickr30k-images') and not os.path.exists('data/raw/flickr30k-images'):
    os.symlink(f'{DRIVE}/flickr30k-images', 'data/raw/flickr30k-images')

!ls -la data/cached data/raw data/splits

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dl-multimodal/cached'

## 5. Confirm GPU and that imports work

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU.')

# Smoke-test the encoders import cleanly (also downloads DistilBERT once, ~250 MB)
import sys; sys.path.insert(0, '.')
from src.encoders.vision_encoder import VisionEncoder
from src.encoders.text_encoder   import TextEncoder
print('encoders OK')

## 6. Train

50 epochs, AdamW lr=1e-3, wd=1e-4, batch=128, learnable temperature. Checkpoints are saved every 10 epochs + final epoch (so 6 total: epoch 10, 20, 30, 40, 50) to keep disk usage under control.

**Sanity check while it runs:** epoch 1 loss should print near `log(128) ≈ 4.85` and drop below 2.0 by epoch 10. By epoch 50 it should be around 1.7–1.9. If it's stuck at 4.5+ after epoch 3, kill it — something's wrong with the data wiring.

In [ ]:
!python src/train.py

## 7. Plot the loss curves

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('experiments/checkpoints/training_history.json'))

plt.figure(figsize=(8, 5))
plt.plot(h['train_loss'], label='train')
plt.plot(h['val_loss'],   label='val')
plt.xlabel('epoch'); plt.ylabel('InfoNCE loss')
plt.title('Training curves — symmetric InfoNCE, learnable temperature')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('experiments/checkpoints/loss_curve.png', dpi=140, bbox_inches='tight')
plt.show()
print(f"final train: {h['train_loss'][-1]:.4f}   final val: {h['val_loss'][-1]:.4f}")

## 8. Build the 31k gallery index for the demo

In [ ]:
!python scripts/build_gallery_index.py \
    --checkpoint experiments/checkpoints/checkpoint_epoch50_learnable.pt

## 9. Evaluate Recall@1 / @5 / @10 on the test set

In [ ]:
!python scripts/evaluate.py \
    --checkpoint experiments/checkpoints/checkpoint_epoch50_learnable.pt

## 10. Copy artefacts back to Drive so you can download them locally

In [ ]:
import shutil, os
OUT = f'{DRIVE}/output'
os.makedirs(OUT, exist_ok=True)

artefacts = [
    'experiments/checkpoints/checkpoint_epoch50_learnable.pt',
    'experiments/checkpoints/training_history.json',
    'experiments/checkpoints/loss_curve.png',
    'data/cached/gallery_embs.pt',
    'data/cached/gallery_ids.json',
]
for src in artefacts:
    if os.path.exists(src):
        shutil.copy(src, OUT)
        print(f'  → {OUT}/{os.path.basename(src)}')
    else:
        print(f'  ! missing: {src}')

print('\nDone. Download from MyDrive/dl-multimodal/output/ to your laptop.')

## Optional — temperature ablation runs

Each of these takes another ~17 min. Skip on the first pass; come back to populate `docs/EXPERIMENTS.md`.

```python
!python src/train.py --temperature 0.05
!python src/train.py --temperature 0.07
!python src/train.py --temperature 0.10
```

Each run saves checkpoints with a different tag — re-run cells 8–9 against each to compare Recall@K.